# Layer Computation Budget

How much computation does each layer use? Norm budget,
information gain, component balance, and residual growth.

## Why This Matters

Layer computation budget characterizes what each layer contributes to the model's computation. Understanding layer-level organization — which layers are critical, which are redundant, and how they specialize — is essential for both interpretability and efficient model design.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.layer_computation_budget import (
    layer_norm_budget, layer_information_gain,
    component_balance, residual_growth_budget,
    computation_budget_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Norm Budget

How much does each layer modify the residual stream?

In [ ]:
result = layer_norm_budget(model, tokens)
print(f"Total budget: {result['total_budget']:.4f}")
for p in result['per_layer']:
    bar = '█' * int(p['fraction'] * 40)
    print(f"  Layer {p['layer']}: total={p['total_norm']:.4f} "
          f"(attn={p['attn_norm']:.4f}, mlp={p['mlp_norm']:.4f}) "
          f"frac={p['fraction']:.3f} {bar}")

## Information Gain

Entropy reduction per layer — which layers add the most useful information?

In [ ]:
result = layer_information_gain(model, tokens, position=-1)
print(f"Most informative: Layer {result['most_informative']}")
for p in result['per_layer']:
    flag = ' [INFO]' if p['is_informative'] else ''
    print(f"  Layer {p['layer']}: H={p['entropy']:.4f}, "
          f"gain={p['info_gain']:+.4f}{flag}")

## Component Balance

Attention vs MLP dominance per layer.

In [ ]:
result = component_balance(model, tokens)
for p in result['per_layer']:
    attn_bar = '█' * int(p['attn_fraction'] * 30)
    mlp_bar = '█' * int(p['mlp_fraction'] * 30)
    print(f"  Layer {p['layer']}: [{p['dominant'].upper():4s}] "
          f"attn={p['attn_fraction']:.3f} {attn_bar}")
    print(f"               mlp={p['mlp_fraction']:.3f} {mlp_bar}")

## Residual Growth

In [ ]:
result = residual_growth_budget(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: {p['pre_norm']:.4f} -> {p['post_norm']:.4f} "
          f"(growth={p['growth']:+.4f}, rate={p['growth_rate']:.3f}x)")

## Budget Summary

In [ ]:
result = computation_budget_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: norm={p['total_norm']:.4f}, "
          f"frac={p['fraction']:.3f}, dom={p['dominant']}")